# Prestador Nota 10 — Regras de Associação (Market Basket Analysis)

Este notebook demonstra a aplicação de **Mineração de Dados** para descobrir regras de associação e dependências entre serviços contratados no marketplace **Prestador Nota 10**.

### Algoritmos utilizados:
- **Apriori**
- **FP-Growth** (*Frequent Pattern Growth*)

### Métricas de Avaliação:
$$\text{Support}(A \rightarrow B) = P(A \cap B)$$
$$\text{Confidence}(A \rightarrow B) = P(B \mid A) = \frac{P(A \cap B)}{P(A)}$$
$$\text{Lift}(A \rightarrow B) = \frac{P(A \cap B)}{P(A) \cdot P(B)}$$

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Carregamento e Preparação dos Dados

In [ ]:
from regras_associacao import carregar_dados_dw, extrair_cestas_compras

df_transacoes = carregar_dados_dw()
cestas = extrair_cestas_compras(df_transacoes)
print(f"Total de transações / cestas agrupadas: {len(cestas)}")
print("Exemplo de cestas:", cestas[:3])

## 2. One-Hot Encoding das Cestas de Serviços

In [ ]:
te = TransactionEncoder()
te_ary = te.fit(cestas).transform(cestas)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
df_encoded.head()

## 3. Mineração de Itemsets Frequentes (FP-Growth vs Apriori)

In [ ]:
min_sup = 0.08
frequent_itemsets = fpgrowth(df_encoded, min_support=min_sup, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
frequent_itemsets.sort_values(by='support', ascending=False).head(10)

## 4. Geração e Análise de Regras de Associação

In [ ]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.15)
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
rules = rules.sort_values(by="lift", ascending=False)
rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage']].head(15)

## 5. Visualização Gráfica das Regras (Suporte x Confiança x Lift)

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    rules['support'], 
    rules['confidence'], 
    c=rules['lift'], 
    cmap='viridis', 
    s=rules['lift'] * 50, 
    alpha=0.8, 
    edgecolors='black'
)
plt.colorbar(scatter, label='Lift')
plt.title('Regras de Associação: Suporte vs Confiança (Tamanho & Cor = Lift)', fontsize=14, fontweight='bold')
plt.xlabel('Suporte', fontsize=12)
plt.ylabel('Confiança', fontsize=12)
plt.tight_layout()
plt.show()